In [ ]:
import os
import sys
import glob
import warnings
import json

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple

sys.path.append(os.path.dirname(__file__))
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Root folder containing one sub-folder per bearing (each sub-folder holds
# the per-minute CSV files, e.g. acc_00001.csv ... acc_00123.csv).
INPUT_PATH = (
    r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets"
    r"\Processed_Data\Downsampled"
)
OUTPUT_PATH = (
    r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets"
    r"\Processed_Data\LSTM_Inputs_v3_OneCondition"
)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Signal processing
XJTU_SAMPLING_RATE: float = 25600.0   # Hz
SEGMENT_LENGTH: int = 1024            # samples per window
STRIDE: int = 512                     # overlapping window stride

# Sliding window sizes for LSTM sequences (in segments, 63 segments = 1 minute).
WINDOW_SIZES: List[int] = [100, 200, 300]

# Scaling
USE_STANDARD_SCALER = True

# Spearman threshold: warn if median |rho| falls below this value.
SPEARMAN_WARN_THRESHOLD: float = 0.30

# CUSUM parameters (conservative)
CUSUM_BASELINE_RATIO: float = 0.20
CUSUM_K_FACTOR:       float = 0.50
CUSUM_H_FACTOR:       float = 8.00
CUSUM_SMOOTHING_WIN:  int   = 63     # rolling mean applied to 63-segment RMS (approx 1 minute)
HI_EMA_ALPHA:         float = 0.1    # EMA smoothing factor for HI before Isotonic Regression

# Cross-validation fold definitions (Focus on 1 condition: 35Hz12kN)
# Condition 1 bearings: Bearing1_1 to Bearing1_5
TARGET_CONDITION = "35Hz12kN"
ALL_BEARINGS: List[str] = [f"Bearing1_{i}" for i in range(1, 6)]
FOLDS: List[Dict[str, List[str]]] = [
    {"val": [f"Bearing1_{i}"]}
    for i in range(1, 6)
]
for _fold in FOLDS:
    _fold["train"] = [b for b in ALL_BEARINGS if b not in _fold["val"]]

# 14 raw feature column names (must match CrossDomainFeatureExtractor output)
RAW_FEATURE_COLS: List[str] = [
    "td_mean", "td_std", "td_rms", "td_peak_value", "td_p2p",
    "td_skewness", "td_kurtosis", "td_peak_factor",
    "td_clearance_factor", "td_shape_factor", "td_impulse_factor",
    "fd_mean_freq", "fd_centroid_freq", "fd_fft_p2p",
]

META_COLS: List[str] = [
    "Bearing_ID", "File_Index", "Segment_Index",
    "Time_Index", "hi_target", "cusum_change_point",
]

print("=" * 65)
print("DATA PREPROCESSING PIPELINE (Single Condition: 35Hz12kN)")
print("=" * 65)
print(f"  Input  : {INPUT_PATH}")
print(f"  Output : {OUTPUT_PATH}")
print(f"  Windows: {WINDOW_SIZES}")
print("=" * 65)

In [ ]:
# ============================================================================
# PHASE 1 — INTRA-FILE RAW FEATURE EXTRACTION
# ============================================================================

def extract_features_from_csv(
    csv_path: str,
    bearing_id: str,
    file_index: int,
) -> pd.DataFrame:
    """
    Extract 14 raw features from a single per-minute CSV file.

    The file is sliced into overlapping 1024-sample windows with 512-sample stride.
    Features are extracted per window from the horizontal (H) vibration channel.
    Outputs exactly 63 rows/segments per CSV file.

    Args:
        csv_path:   Absolute path to the per-minute CSV file.
        bearing_id: Canonical bearing identifier (e.g., "Bearing1_3").
        file_index: 1-based integer index of the CSV file in the bearing's
                    chronological sequence.

    Returns:
        DataFrame of shape (63, 14+meta) or None on failure.
    """
    try:
        df_raw = pd.read_csv(csv_path, header=None)
    except Exception as exc:
        print(f"    [WARNING] Cannot read {csv_path}: {exc}")
        return None

    # XJTU-SY format: column 0 = horizontal, column 1 = vertical.
    if df_raw.shape[1] < 1:
        return None

    # Coerce to numeric to safely ignore header strings
    h_series = pd.to_numeric(df_raw.iloc[:, 0], errors='coerce')
    h_signal = h_series.dropna().to_numpy(dtype=np.float64)
    n_total   = len(h_signal)

    # For 32768 samples, window=1024, stride=512 -> exactly 63 segments
    if n_total < SEGMENT_LENGTH:
        return None

    extractor = CrossDomainFeatureExtractor(sampling_rate=XJTU_SAMPLING_RATE)

    # Use stride of 512 (which is SEGMENT_LENGTH // 2)
    stride = 512
    n_windows = (n_total - SEGMENT_LENGTH) // stride + 1
    
    # Cap to exactly 63 to avoid uneven sizes if file has extra samples
    n_windows = min(n_windows, 63)

    rows = []
    for w in range(n_windows):
        start = w * stride
        seg   = h_signal[start: start + SEGMENT_LENGTH]
        feats = extractor.extract_all_features(seg)
        feats["Bearing_ID"]  = bearing_id
        feats["File_Index"]  = file_index
        feats["Segment_Index"] = w
        rows.append(feats)

    if not rows:
        return None

    return pd.DataFrame(rows)

In [ ]:
# ============================================================================
# PHASE 2 — CONCATENATE SEGMENTS (NO AGGREGATION)
# ============================================================================

def run_phase1_and_phase2(
    bearing_csv_dir: str,
    bearing_id: str,
) -> Optional[pd.DataFrame]:
    """
    Execute Phases 1 and 2 for a single bearing.

    Discovers all per-minute CSV files, extracts 63 segments per file,
    and concatenates them into a continuous time-series DataFrame.

    Args:
        bearing_csv_dir: Directory containing the per-minute CSV files.
        bearing_id:      Canonical bearing identifier.

    Returns:
        Segment-level DataFrame or None if no data found.
    """
    csv_files = sorted(glob.glob(os.path.join(bearing_csv_dir, "*.csv")))
    if not csv_files:
        print(f"  [WARNING] No CSV files in {bearing_csv_dir}")
        return None

    all_segment_rows: List[pd.DataFrame] = []

    for file_idx, csv_path in enumerate(csv_files, start=1):
        df_windows = extract_features_from_csv(csv_path, bearing_id, file_idx)
        if df_windows is None or df_windows.empty:
            continue
        all_segment_rows.append(df_windows)

    if not all_segment_rows:
        return None

    df_bearing = pd.concat(all_segment_rows, ignore_index=True)
    # Create an absolute global segment index (Time_Index)
    df_bearing["Time_Index"] = np.arange(len(df_bearing), dtype=float)
    df_bearing = df_bearing.sort_values("Time_Index").reset_index(drop=True)
    df_bearing = df_bearing.fillna(0.0)

    print(
        f"  {bearing_id}: {len(csv_files)} files -> "
        f"{len(df_bearing)} segments (63 segments/min)"
    )
    return df_bearing

In [ ]:
# ============================================================================
# PHASE 3 — CAUSAL GROUND TRUTH CONSTRUCTION
# ============================================================================

def construct_causal_hi(df_segments: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Construct piecewise-linear Health Index (HI) targets on the 63-segment/min series.

    Execution order is strictly causal:
    1. CUSUM is applied to the smoothed RMS column.
    2. HI target is derived from the detected change point (piecewise linear).
    3. HI target is smoothed with EMA.
    4. Isotonic Regression is applied to force strict monotonically decreasing labels.

    Args:
        df_segments: Segment-level DataFrame with RAW_FEATURE_COLS columns.

    Returns:
        Tuple of (df with hi_target + cusum_change_point columns, metadata dict).
    """
    df_out      = df_segments.copy()
    n_total     = len(df_out)
    bearing_id  = df_out["Bearing_ID"].iloc[0]

    rms_col = "td_rms"
    if rms_col not in df_out.columns:
        raise ValueError(f"Column '{rms_col}' not found in {bearing_id}.")

    # 1. Smooth RMS using rolling window (approx 1 minute = 63 segments)
    smoothed_rms = (
        df_out[rms_col]
        .rolling(window=CUSUM_SMOOTHING_WIN, min_periods=1, center=True)
        .mean()
        .values
    )

    # 2. CUSUM
    detector = UnivariateCUSUMDetector(
        baseline_ratio=CUSUM_BASELINE_RATIO,
        k_factor=CUSUM_K_FACTOR,
        h_factor=CUSUM_H_FACTOR,
    )
    change_point, _ = detector.fit_predict(smoothed_rms)

    # 3. Piecewise linear HI
    hi = np.ones(n_total, dtype=np.float64)
    degradation_len = n_total - change_point
    if degradation_len > 1:
        hi[change_point:] = np.linspace(1.0, 0.0, degradation_len)
    elif degradation_len == 1:
        hi[change_point] = 0.0
        
    # 4. Smooth HI with EMA
    hi_ema = pd.Series(hi).ewm(alpha=HI_EMA_ALPHA, adjust=False).mean().values
    
    # 5. Monotonicity via Isotonic Regression (increasing=False)
    iso_reg = IsotonicRegression(increasing=False, out_of_bounds='clip')
    hi_iso = iso_reg.fit_transform(np.arange(n_total), hi_ema)

    df_out["hi_target"] = hi_iso
    df_out["cusum_change_point"] = change_point

    meta = {
        "Bearing_ID": bearing_id,
        "Total_Segments": n_total,
        "Total_Minutes": n_total / 63.0,
        "Change_Point_Idx": change_point,
        "Change_Point_Min": change_point / 63.0,
        "Degradation_Len": degradation_len,
    }
    return df_out, meta

In [ ]:
# ============================================================================
# PHASE 4 — MANDATORY VISUAL VALIDATION (SPEARMAN RANK CORRELATION)
# ============================================================================

def validate_features(
    df_gt: pd.DataFrame,
    bearing_id: str,
    output_dir: str,
) -> float:
    """
    Computes Spearman rank correlation between all 14 features and the HI target.

    Emits a WARNING banner to stdout if the median absolute Spearman rho
    falls below SPEARMAN_WARN_THRESHOLD.

    Args:
        df_gt:      DataFrame with RAW_FEATURE_COLS and 'hi_target'.
        bearing_id: Identifier used in plot titles and filenames.
        output_dir: Directory to save the validation plot.

    Returns:
        Median absolute Spearman rho across all 14 features.
    """
    hi     = df_gt["hi_target"].values
    # Convert Time_Index to Minutes for plotting
    time_min = df_gt["Time_Index"].values / 63.0
    cp_min = df_gt['cusum_change_point'].iloc[0] / 63.0

    rho_records = []
    for feat in RAW_FEATURE_COLS:
        if feat not in df_gt.columns:
            continue
        rho, pval = scipy_stats.spearmanr(df_gt[feat].values, hi)
        rho_records.append({"feature": feat, "rho": rho, "pval": pval})

    rho_df = pd.DataFrame(rho_records).sort_values("rho", key=abs, ascending=False)
    median_abs_rho = float(rho_df["rho"].abs().median())

    print(f"\n  [Phase 4 — {bearing_id}] Spearman Rank vs HI Target (14 Features)")
    print(f"  {'Feature':<30} {'rho':>8}  {'p-value':>10}")
    print("  " + "-" * 53)
    for _, r in rho_df.iterrows():
        print(f"  {r['feature']:<30} {r['rho']:>8.4f}  {r['pval']:>10.4e}")
    print(f"\n  Median |rho| = {median_abs_rho:.4f}")

    if median_abs_rho < SPEARMAN_WARN_THRESHOLD:
        warnings.warn(
            f"\n{'='*65}\n"
            f"  VALIDATION WARNING — {bearing_id}\n"
            f"  Median |Spearman rho| = {median_abs_rho:.4f} "
            f"< threshold {SPEARMAN_WARN_THRESHOLD}.\n"
            f"{'='*65}",
            UserWarning,
            stacklevel=2,
        )

    # --- Plot ---
    n_feats  = len(RAW_FEATURE_COLS)
    n_cols   = 4
    n_rows   = (n_feats + n_cols - 1) // n_cols + 1  # +1 row for HI overlay

    fig = plt.figure(figsize=(n_cols * 5, n_rows * 3))
    gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig)

    # Top full-width: HI target curve
    ax_hi = fig.add_subplot(gs[0, :])
    ax_hi.plot(time_min, hi, color="#c0392b", linewidth=2, label="HI Target (Isotonic)")
    ax_hi.set_title(
        f"{bearing_id} — Health Index Target  "
        f"(CUSUM CP = {cp_min:.2f} min)"
    )
    ax_hi.set_xlabel("Time (Minutes)")
    ax_hi.set_ylabel("HI")
    ax_hi.legend()
    ax_hi.grid(True, linestyle="--", alpha=0.5)

    # Remaining rows: one subplot per feature
    for feat_idx, feat in enumerate(RAW_FEATURE_COLS):
        row = (feat_idx // n_cols) + 1
        col = feat_idx % n_cols
        ax  = fig.add_subplot(gs[row, col])

        rho_row = rho_df[rho_df["feature"] == feat]
        rho_val = rho_row["rho"].values[0] if len(rho_row) > 0 else float("nan")

        ax.plot(time_min, df_gt[feat].values, linewidth=1.0, alpha=0.85)
        ax.set_title(f"{feat}\nrho={rho_val:.3f}", fontsize=8)
        ax.set_xlabel("Time (min)", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.grid(True, linestyle="--", alpha=0.4)

    plt.suptitle(
        f"Feature-HI Validation — {bearing_id}  "
        f"(Median |rho| = {median_abs_rho:.3f})",
        fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    plot_path = os.path.join(output_dir, f"validation_{bearing_id}.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  [Phase 4] Validation plot saved: {plot_path}")

    csv_path = os.path.join(output_dir, f"spearman_{bearing_id}.csv")
    rho_df.to_csv(csv_path, index=False)

    return median_abs_rho

In [ ]:
# ============================================================================
# PHASE 5 — GLOBAL SCALING (StandardScaler)
# ============================================================================

def scale_features(
    df_train_list: List[pd.DataFrame],
    df_val_list:   List[pd.DataFrame],
) -> Tuple[List[pd.DataFrame], List[pd.DataFrame], StandardScaler]:
    """
    Apply global StandardScaler to the 14 raw features.
    
    CRITICAL: Fit ONLY on training data, then transform both train and val.
    Never fit per-bearing or globally before split (Data Leakage).
    
    Args:
        df_train_list: List of training DataFrames.
        df_val_list:   List of validation DataFrames.
        
    Returns:
        scaled_train_list, scaled_val_list, fitted_scaler
    """
    scaler = StandardScaler() if USE_STANDARD_SCALER else MinMaxScaler()
    
    # Collect all train data to fit
    train_concat = pd.concat(df_train_list, ignore_index=True)
    scaler.fit(train_concat[RAW_FEATURE_COLS])
    
    scaled_train = []
    for df in df_train_list:
        df_scaled = df.copy()
        df_scaled[RAW_FEATURE_COLS] = scaler.transform(df[RAW_FEATURE_COLS])
        scaled_train.append(df_scaled)
        
    scaled_val = []
    for df in df_val_list:
        df_scaled = df.copy()
        df_scaled[RAW_FEATURE_COLS] = scaler.transform(df[RAW_FEATURE_COLS])
        scaled_val.append(df_scaled)
        
    return scaled_train, scaled_val, scaler

In [ ]:
# ============================================================================
# PHASE 6 — SLIDING WINDOWS + NATIVE 3D .npy SAVING
# ============================================================================

def create_3d_windows(
    df_scaled: pd.DataFrame,
    window_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create 3D sliding windows from scaled features.

    Args:
        df_scaled:   Scaled DataFrame with RAW_FEATURE_COLS and 'hi_target'.
        window_size: Number of consecutive segments per window.

    Returns:
        Tuple (X, y) where:
            X shape: (N_windows, window_size, 14)
            y shape: (N_windows,)  — HI value of the last segment in window.
    """
    present_feats = [c for c in RAW_FEATURE_COLS if c in df_scaled.columns]
    feature_matrix = df_scaled[present_feats].values.astype(np.float32)
    hi_values      = df_scaled["hi_target"].values.astype(np.float32)

    n_samples = len(feature_matrix)
    if n_samples <= window_size:
        return np.empty((0, window_size, len(present_feats)), dtype=np.float32), \
               np.empty((0,), dtype=np.float32)

    n_windows = n_samples - window_size + 1
    X = np.lib.stride_tricks.sliding_window_view(
        feature_matrix, window_shape=window_size, axis=0
    )  # Shape: (n_windows, n_features, window_size)
    X = X.transpose(0, 2, 1)  # -> (n_windows, window_size, n_features)
    y = hi_values[window_size - 1:]

    assert X.shape == (n_windows, window_size, len(present_feats)), \
        f"Unexpected X shape: {X.shape}"
    assert y.shape == (n_windows,), f"Unexpected y shape: {y.shape}"

    return X.astype(np.float32), y.astype(np.float32)


def save_npy(array: np.ndarray, path: str) -> None:
    """Save a numpy array as a .npy file, creating parent directories."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, array)
    print(f"  [Saved] {path}  shape={array.shape}")

In [ ]:
# ============================================================================
# MAIN ORCHESTRATOR
# ============================================================================

def discover_bearing_dirs(root: str) -> Dict[str, str]:
    """
    Walk the dataset root and map bearing IDs to their CSV directories.

    Returns:
        Dict mapping bearing_id (e.g. "Bearing1_3") to its directory path.
    """
    mapping: Dict[str, str] = {}
    condition_map = {"35": 1, "37": 2, "40": 3}

    for cond_dir in sorted(os.listdir(root)):
        cond_path = os.path.join(root, cond_dir)
        if not os.path.isdir(cond_path):
            continue
        c_idx = None
        for key, val in condition_map.items():
            if key in cond_dir:
                c_idx = val
                break
        if c_idx is None:
            continue
        for bearing_dir in sorted(os.listdir(cond_path)):
            b_path = os.path.join(cond_path, bearing_dir)
            if not os.path.isdir(b_path):
                continue
            # Extract bearing number from folder name (e.g. "Bearing1" -> "1")
            digits = "".join(filter(str.isdigit, bearing_dir))
            b_num  = digits[-1] if digits else "1"
            b_id   = f"Bearing{c_idx}_{b_num}"
            mapping[b_id] = b_path

    return mapping


def run_pipeline() -> None:
    """
    Main preprocessing pipeline orchestrator.

    Execution sequence:
        1. Discover bearing directories.
        2. Phase 1 + 2: intra-file feature extraction and aggregation.
        3. Phase 3: causal CUSUM + piecewise-linear HI target construction.
        4. Phase 4: mandatory Spearman visual validation (with halt warning).
        5. Phase 5: global Scaling (fit on train, transform on val/test).
        6. Phase 6: sliding windows, 3D assertion, native .npy saving.
    """
    bearing_dirs = discover_bearing_dirs(INPUT_PATH)
    if not bearing_dirs:
        print(f"ERROR: No bearing directories found under {INPUT_PATH}")
        return

    print(f"\nFound {len(bearing_dirs)} bearing directories.")

    # ------------------------------------------------------------------
    # Phases 1 & 2: Extract + Aggregate per bearing
    # ------------------------------------------------------------------
    print("\n--- Phase 1 + 2: Feature Extraction and Segment Concatenation ---")
    all_aggregated: Dict[str, pd.DataFrame] = {}

    # Only process target condition for now
    target_bearings = {k: v for k, v in bearing_dirs.items() if k in ALL_BEARINGS}

    for b_id, b_dir in tqdm(target_bearings.items(), desc="Bearings"):
        df_agg = run_phase1_and_phase2(b_dir, b_id)
        if df_agg is not None and not df_agg.empty:
            all_aggregated[b_id] = df_agg

    if not all_aggregated:
        print("ERROR: No aggregated data produced. Check INPUT_PATH and CSV format.")
        return

    # ------------------------------------------------------------------
    # Phase 3: Causal ground truth construction
    # ------------------------------------------------------------------
    print("\n--- Phase 3: Causal CUSUM + HI Target Construction ---")
    all_labeled: Dict[str, pd.DataFrame] = {}
    all_metadata = []

    for b_id, df_agg in all_aggregated.items():
        df_gt, meta = construct_causal_hi(df_agg)
        all_labeled[b_id] = df_gt
        all_metadata.append(meta)

    meta_df = pd.DataFrame(all_metadata)
    meta_path = os.path.join(OUTPUT_PATH, "bearing_metadata.csv")
    meta_df.to_csv(meta_path, index=False)
    print(f"\nMetadata saved: {meta_path}")

    # ------------------------------------------------------------------
    # Phase 4: Mandatory visual validation (Spearman)
    # ------------------------------------------------------------------
    print("\n--- Phase 4: Mandatory Visual Validation (Spearman Correlation) ---")
    validation_dir = os.path.join(OUTPUT_PATH, "validation_plots")
    validation_summary = []

    for b_id, df_gt in all_labeled.items():
        median_rho = validate_features(df_gt, b_id, validation_dir)
        validation_summary.append({"bearing": b_id, "median_abs_rho": median_rho})

    val_df = pd.DataFrame(validation_summary)
    val_path = os.path.join(validation_dir, "spearman_summary.csv")
    val_df.to_csv(val_path, index=False)
    print(f"\nSpearman summary saved: {val_path}")

    overall_poor = val_df[val_df["median_abs_rho"] < SPEARMAN_WARN_THRESHOLD]
    if not overall_poor.empty:
        print(
            "\n  *** HALT RECOMMENDATION ***\n"
            "  The following bearings have poor feature-HI correlation:\n"
            f"{overall_poor.to_string(index=False)}\n"
            "  Investigate feature engineering before proceeding to training."
        )

    # ------------------------------------------------------------------
    # Phases 5 & 6: Scaling, windowing, 3D .npy saving per fold
    # ------------------------------------------------------------------
    print("\n--- Phase 5 + 6: Scaling, Windowing, and 3D .npy Saving ---")

    for ws in WINDOW_SIZES:
        print(f"\n  Window size = {ws} ({ws / 63.0:.1f} min physical context)")
        ws_dir = os.path.join(OUTPUT_PATH, f"ws_{ws}")
        os.makedirs(ws_dir, exist_ok=True)

        for fold_idx, fold in enumerate(FOLDS, start=1):
            train_ids = [b for b in fold["train"] if b in all_labeled]
            val_ids   = [b for b in fold["val"] if b in all_labeled]

            if not train_ids or not val_ids:
                print(f"  [WARNING] Fold {fold_idx} WS={ws}: empty split. Skipping.")
                continue

            # Prepare list of Dataframes
            train_df_list = [all_labeled[b] for b in train_ids]
            val_df_list   = [all_labeled[b] for b in val_ids]

            # Scale Globally based on Train data
            scaled_train_list, scaled_val_list, _ = scale_features(train_df_list, val_df_list)

            # Create 3D Windows
            train_X_parts, train_y_parts = [], []
            val_X_parts,   val_y_parts   = [], []
            
            for df_scaled in scaled_train_list:
                X, y = create_3d_windows(df_scaled, ws)
                if X.shape[0] > 0:
                    train_X_parts.append(X)
                    train_y_parts.append(y)

            for df_scaled in scaled_val_list:
                X_v, y_v = create_3d_windows(df_scaled, ws)
                if X_v.shape[0] > 0:
                    val_X_parts.append(X_v)
                    val_y_parts.append(y_v)

            X_train = np.concatenate(train_X_parts, axis=0)
            y_train = np.concatenate(train_y_parts, axis=0)
            X_val   = np.concatenate(val_X_parts,   axis=0)
            y_val   = np.concatenate(val_y_parts,   axis=0)

            # Shuffle training set globally
            rng  = np.random.default_rng(seed=42)
            perm = rng.permutation(len(X_train))
            X_train, y_train = X_train[perm], y_train[perm]

            fold_dir = os.path.join(ws_dir, f"fold_{fold_idx}")
            save_npy(X_train, os.path.join(fold_dir, "X_train.npy"))
            save_npy(y_train, os.path.join(fold_dir, "y_train.npy"))
            save_npy(X_val,   os.path.join(fold_dir, "X_val.npy"))
            save_npy(y_val,   os.path.join(fold_dir, "y_val.npy"))

            print(
                f"  Fold {fold_idx} WS={ws}: "
                f"train={X_train.shape}, val={X_val.shape}"
            )

    print("\n" + "=" * 65)
    print("PREPROCESSING COMPLETE — all .npy files written to:")
    print(f"  {OUTPUT_PATH}")
    print("=" * 65)


# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    run_pipeline()